# Final Dataset Construction

This notebook combines all engineered features produced during the data preparation phase.

Sources:
- listings_features.csv
- review_features.csv

The resulting dataset will be used for machine learning models such as Decision Trees and Random Forests.

In [33]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: /Users/santiagotawata/Desktop/airbnb-rome-analysis


In [34]:
listings_features = pd.read_csv(
    "../data/listings_features.csv"
)

review_features = pd.read_csv(
    "../data/review_features.csv"
)

## Inspect Input Datasets

In [35]:
print(listings_features.shape)
print(review_features.shape)

(34324, 58)
(32827, 4)


In [36]:
listings_features.head()

,id,price,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,bathrooms,...,neighbourhood_cleansed_VII San Giovanni/Cinecittà,neighbourhood_cleansed_VIII Appia Antica,neighbourhood_cleansed_X Ostia/Acilia,neighbourhood_cleansed_XI Arvalia/Portuense,neighbourhood_cleansed_XII Monte Verde,neighbourhood_cleansed_XIII Aurelia,neighbourhood_cleansed_XIV Monte Mario,neighbourhood_cleansed_XV Cassia/Flaminia,distance_to_colosseum,location_cluster
0,2737,59.52,41.87136,12.48215,Private Room,Private room,1,1.0,1.0,1.5,...,False,True,False,False,False,False,False,False,2.252726,2.0
1,12398,117.13,41.92582,12.46928,Apartment,Entire home/apt,3,2.0,3.0,1.0,...,False,False,False,False,False,False,False,False,4.389669,0.0
2,19965,160.60,41.90823,12.45293,Apartment,Entire home/apt,5,2.0,3.0,1.0,...,False,False,False,False,False,False,False,False,3.824847,0.0
3,20534,243.67,41.88992,12.46823,Entire Home,Entire home/apt,4,1.0,3.0,1.0,...,False,False,False,False,False,False,False,False,1.989591,2.0
4,20587,302.50,41.88992,12.46823,Entire Home,Entire home/apt,4,2.0,4.0,1.0,...,False,False,False,False,False,False,False,False,1.989591,2.0


In [37]:
review_features.head()

,listing_id,review_count,avg_review_length,latest_review_date
0,2737,5,51.800000,2015-05-08
1,11834,302,74.165563,2026-06-07
2,12398,85,84.658824,2025-08-01
3,19965,195,44.953846,2026-06-13
4,20534,50,60.200000,2022-11-22


In [38]:
## Verify Merge Key

In [39]:
print("id" in listings_features.columns)
print("listing_id" in review_features.columns)

True
True


## Merge Listings and Review Features

Review-based variables are merged into the listings dataset using the listing identifier.

A left join is used to preserve all listings.

In [40]:
print(review_features.columns.tolist())

['listing_id', 'review_count', 'avg_review_length', 'latest_review_date']


In [41]:
final_df = listings_features.merge(
    review_features,
    left_on="id",
    right_on="listing_id",
    how="left"
)

#deletion of duplicate key
final_df.drop(
    columns=["listing_id"],
    inplace=True
)

## Handle Missing Review Features

Listings without reviews do not appear in the reviews dataset.

Missing review-derived variables are replaced with zero.

In [42]:
final_df["review_count"] = (
    final_df["review_count"]
    .fillna(0)
)

final_df["avg_review_length"] = (
    final_df["avg_review_length"]
    .fillna(0)
)

In [43]:
final_df.columns.tolist()

['id',
 'price',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bedrooms',
 'beds',
 'bathrooms',
 'host_is_superhost',
 'host_response_rate',
 'host_acceptance_rate',
 'review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_location',
 'review_scores_value',
 'number_of_reviews',
 'reviews_per_month',
 'minimum_nights',
 'maximum_nights',
 'instant_bookable',
 'beds_per_guest',
 'bathrooms_per_guest',
 'amenities_count',
 'has_wifi',
 'has_air_conditioning',
 'has_kitchen',
 'has_washer',
 'has_dryer',
 'has_parking',
 'has_elevator',
 'has_tv',
 'has_workspace',
 'professional_host',
 'review_quality_index',
 'listing_age_days',
 'review_recency_days',
 'review_intensity',
 'occupancy_rate',
 'demand_proxy',
 'neighbourhood_cleansed_I Centro Storico',
 'neighbourhood_cleansed_II Parioli/Nomentano',
 'neighbourhood_cleansed_III Monte Sacro',
 'neighbourhood_cleansed_IV Tiburtina',
 'neighbourhood_cleansed_IX Eur',
 'neighbourhood_cleansed_V

In [44]:
final_df["latest_review_date"] = pd.to_datetime(
    final_df["latest_review_date"]
)

`days_since_latest_review`

In [45]:
reference_date = final_df["latest_review_date"].max()

final_df["days_since_latest_review"] = (
    reference_date - final_df["latest_review_date"]
).dt.days

Full fill empty dates

In [46]:
final_df["days_since_latest_review"] = (
    final_df["days_since_latest_review"]
    .fillna(9999)
)

## Final Dataset Validation

In [47]:
print(final_df.shape)

(34324, 62)


In [48]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34324 entries, 0 to 34323
Data columns (total 62 columns):
 #   Column                                             Non-Null Count  Dtype         
---  ------                                             --------------  -----         
 0   id                                                 34324 non-null  int64         
 1   price                                              34324 non-null  float64       
 2   latitude                                           34324 non-null  float64       
 3   longitude                                          34324 non-null  float64       
 4   property_type                                      34324 non-null  object        
 5   room_type                                          34324 non-null  object        
 6   accommodates                                       34324 non-null  int64         
 7   bedrooms                                           34324 non-null  float64       
 8   beds            

In [49]:
final_df.isnull().sum().sort_values(
    ascending=False
).head(20)

host_response_rate                                   34324
host_acceptance_rate                                 34324
latest_review_date                                    3753
location_cluster                                      3337
maximum_nights                                          13
minimum_nights                                          13
neighbourhood_cleansed_IX Eur                            0
neighbourhood_cleansed_IV Tiburtina                      0
neighbourhood_cleansed_III Monte Sacro                   0
neighbourhood_cleansed_II Parioli/Nomentano              0
id                                                       0
neighbourhood_cleansed_I Centro Storico                  0
occupancy_rate                                           0
review_intensity                                         0
review_recency_days                                      0
listing_age_days                                         0
review_quality_index                                    

## Export Final Dataset

The final modeling dataset is saved and will be used for machine learning models.

In [50]:
final_df.to_csv(
    "../data/final_dataset.csv",
    index=False
)

print("final_dataset.csv saved")

final_dataset.csv saved


In [51]:
final_df.shape

(34324, 62)

In [52]:
final_df.columns.tolist()

['id',
 'price',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bedrooms',
 'beds',
 'bathrooms',
 'host_is_superhost',
 'host_response_rate',
 'host_acceptance_rate',
 'review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_location',
 'review_scores_value',
 'number_of_reviews',
 'reviews_per_month',
 'minimum_nights',
 'maximum_nights',
 'instant_bookable',
 'beds_per_guest',
 'bathrooms_per_guest',
 'amenities_count',
 'has_wifi',
 'has_air_conditioning',
 'has_kitchen',
 'has_washer',
 'has_dryer',
 'has_parking',
 'has_elevator',
 'has_tv',
 'has_workspace',
 'professional_host',
 'review_quality_index',
 'listing_age_days',
 'review_recency_days',
 'review_intensity',
 'occupancy_rate',
 'demand_proxy',
 'neighbourhood_cleansed_I Centro Storico',
 'neighbourhood_cleansed_II Parioli/Nomentano',
 'neighbourhood_cleansed_III Monte Sacro',
 'neighbourhood_cleansed_IV Tiburtina',
 'neighbourhood_cleansed_IX Eur',
 'neighbourhood_cleansed_V

## Data Preprocessing

Machine learning algorithms require numerical inputs.

Categorical variables are encoded and date variables are transformed into numerical representations.

dsjnfkcjds,